In [17]:
##### Compiles csv of training data for regions in dataset and does final cleaning

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [18]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Vectors')

# intensities 
capital = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
labor = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")

# lat / lon 
lat_lon = pd.read_csv(f"{cd}/Data/Clean/Predictors/Vectors/lat_lon.csv")

# save path
save_path_capital = f'{cd}/Data/Clean/Training_data/capital.csv'
save_path_labor = f'{cd}/Data/Clean/Training_data/labor.csv'

In [19]:
##### Merge all predictors into one df (dropping all duplicate PROJ_ID entries first)
dfs = []
for f in os.listdir(predictors):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(predictors, f))
        dupes = df['PROJ_ID'].duplicated().sum()
        if dupes > 0:
            df = df.drop_duplicates(subset='PROJ_ID')
        dfs.append(df)

predictors_df = reduce(lambda left, right: pd.merge(left, right, on='PROJ_ID', how='outer'), dfs)

In [20]:
### Merge with capital and labor 

### Capital
capital = capital.drop_duplicates(subset='PROJ_ID')
capital_merge = capital.merge(predictors_df, on='PROJ_ID', how='left')

### Labor
labor = labor.drop_duplicates(subset='PROJ_ID')
labor_merge = labor.merge(predictors_df, on='PROJ_ID', how='left')

In [21]:
### Save
capital_merge.to_csv(save_path_capital, index=False)
labor_merge.to_csv(save_path_labor, index=False)

In [22]:
capital_merge.columns

Index(['PROJ_ID', 'GEO_ID_NAME', 'ag_capital_stock_USD_nominal',
       'total_production_USD', 'total_production_t',
       'capital_intensity_USD_per_USD', 'capital_intensity_USD_per_tonne',
       'SOC', 'pct_cropland_irrigated', 'female_share',
       'pop_density_people_per_km2', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'growing_season_length_days',
       'child_dependency_ratio', 'GDP_pc', 'USD_production_per_HA',
       'tonnes_production_per_HA', 'slope', 'buffalo_density_per_km2',
       'chicken_density_per_km2', 'cattle_density_per_km2',
       'goats_density_per_km2', 'pigs_density_per_km2',
       'sheep_density_per_km2', 'livestock_density_LU_per_km2',
       'cereals_share_production_USD', 'fibres_share_production_USD',
       'fruits_share_production_USD', 'oilcrops_share_production_USD',
       'pulses_share_production_USD', 'roots_tubers_share_production